# Grad-CAM interpretability: where do WEAPD models look on field frost images?
Answers the interpretability recommendation of Vallejo et al. (2026, Biosystems Engineering).
1. Grad-CAM of the **cold-protection score** (p(frost)+p(rime)+p(glaze)+p(snow)) for WEAPD test images and EV3 vineyard images.
2. **Quantitative:** share of Grad-CAM energy that falls on annotated plant pixels (EV3 masks) vs the plant's share of the image
   (attention ratio > 1 = model looks at the plant more than chance; < 1 = background/sky/soil).
3. Same analysis after **fine-tuning on EV3 field images** (pool split), to test whether recalibration moves attention to the plant.

**Inputs:** `ev3-frost-grape`, WEAPD (`jehanbhathena/weather-dataset`), output of the **EV3 notebook** (`notebook4873e635e9`,
contains `ev3_results/weapd_<model>.pt` and `ev3_split.csv`) and `notebook0728dabbd1` (v2 `split.csv`).
**Accelerator: GPU T4 x2, Internet On.** Download `gradcam_results_small.zip`.

In [ ]:
CFG = dict(BACKBONES=["swin_tiny_patch4_window7_224", "convnext_tiny"], COLD=["frost", "rime", "glaze", "snow"],
           N_SHOW=6, FT_EPOCHS=12, FT_LR=1e-4, SEED=0, IMG_RESIZE=256, IMG_SIZE=224, OUT="/kaggle/working/gradcam_results")
import os, glob, json, random
import numpy as np, pandas as pd, torch, torch.nn.functional as F, timm, torchvision.transforms as TV
from PIL import Image, ImageDraw, ImageFile
from scipy.stats import wilcoxon
ImageFile.LOAD_TRUNCATED_IMAGES = True; os.makedirs(CFG["OUT"], exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
random.seed(CFG["SEED"]); np.random.seed(CFG["SEED"]); torch.manual_seed(CFG["SEED"])
MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
norm = TV.Compose([TV.ToTensor(), TV.Normalize(MEAN, STD)])
R, S = CFG["IMG_RESIZE"], CFG["IMG_SIZE"]; o = (R - S) // 2
def prep(im): return im.convert("RGB").resize((R, R), Image.BICUBIC).crop((o, o, o + S, o + S))   # same geometry as training
print(DEVICE)

In [ ]:
# 1. Inputs
ann = glob.glob("/kaggle/input/**/_annotations.coco.json", recursive=True); assert ann, "add ev3-frost-grape"
EV3_ROOT = os.path.dirname(os.path.dirname(ann[0]))
SPLIT_CSV = next(iter(glob.glob("/kaggle/input/**/results_v2/split.csv", recursive=True)), None)
EV3_SPLIT = next(iter(glob.glob("/kaggle/input/**/ev3_split.csv", recursive=True)), None)
CKPT = {b: next(iter(glob.glob(f"/kaggle/input/**/weapd_{b}.pt", recursive=True) + glob.glob(f"/kaggle/input/**/best_{b}_s0.pt", recursive=True)), None)
        for b in CFG["BACKBONES"]}
WEAPD_NAMES = {"dew", "fogsmog", "frost", "glaze", "hail", "lightning", "rain", "rainbow", "rime", "sandstorm", "snow"}
WEAPD_ROOT = None
for d, sub, _ in os.walk("/kaggle/input", followlinks=True):
    if len(WEAPD_NAMES & {s.lower() for s in sub}) >= 9 and "cache" not in d: WEAPD_ROOT = d; break
print("EV3", EV3_ROOT, "\nsplit", SPLIT_CSV, "\nev3 split", EV3_SPLIT, "\nckpt", CKPT, "\nWEAPD", WEAPD_ROOT)
assert all(CKPT.values()), "attach the EV3 notebook output (weapd_<model>.pt)"
wd = pd.read_csv(SPLIT_CSV, dtype={"phash": str}); CLASSES = sorted(wd.label.unique()); C2I = {c: i for i, c in enumerate(CLASSES)}
COLD = [C2I[c] for c in CFG["COLD"]]

In [ ]:
# 2. EV3 images, labels and plant masks (any annotated polygon = plant)
rows = []
for s in ["train", "valid", "test"]:
    j = json.load(open(os.path.join(EV3_ROOT, s, "_annotations.coco.json"))); cats = {c["id"]: c["name"] for c in j["categories"]}
    by = {}
    for a in j["annotations"]: by.setdefault(a["image_id"], []).append(a)
    for im in j["images"]:
        A = by.get(im["id"], []); m = Image.new("L", (im["width"], im["height"]), 0); dr = ImageDraw.Draw(m)
        for a in A:
            for poly in (a.get("segmentation") or []):
                if isinstance(poly, list) and len(poly) >= 6: dr.polygon(poly, fill=255)
        rows.append(dict(file=os.path.join(EV3_ROOT, s, im["file_name"]), y=int(any(cats[a["category_id"]].startswith("frozen") for a in A)),
                         mask=np.asarray(prep(m.convert("RGB")).convert("L")) > 127))
ev = pd.DataFrame(rows)
if EV3_SPLIT:
    sp = pd.read_csv(EV3_SPLIT); sp["k"] = sp.file.str.split("/").str[-2:].str.join("/"); ev["k"] = ev.file.str.split("/").str[-2:].str.join("/")
    ev = ev.merge(sp[["k", "split"]], on="k", how="left")
else:
    ev["split"] = np.where(np.random.rand(len(ev)) < 0.5, "pool", "test")
ev["plant_frac"] = [m.mean() for m in ev["mask"]]
print(ev.split.value_counts().to_dict(), "mean plant fraction", ev.plant_frac.mean().round(3))

In [ ]:
# 3. Grad-CAM on the final feature map (works for NCHW ConvNeXt and NHWC Swin features in timm)
def gradcam(model, x, target_fn):
    model.eval(); model.zero_grad(set_to_none=True)
    feat = model.forward_features(x); feat.retain_grad()
    logits = model.forward_head(feat); score = target_fn(logits); score.sum().backward()
    f, g = feat.detach(), feat.grad.detach()
    if f.shape[1] != f.shape[-1] and f.shape[-1] > f.shape[1]: f, g = f.permute(0, 3, 1, 2), g.permute(0, 3, 1, 2)   # NHWC -> NCHW
    w = g.mean(dim=(2, 3), keepdim=True); cam = F.relu((w * f).sum(1, keepdim=True))
    cam = F.interpolate(cam, size=x.shape[-2:], mode="bilinear", align_corners=False)[:, 0]
    cam = cam / (cam.flatten(1).max(1)[0].view(-1, 1, 1) + 1e-8)
    return cam.cpu().numpy(), torch.softmax(logits, 1).detach().cpu().numpy()
cold_target = lambda lg: torch.softmax(lg, 1)[:, COLD].sum(1)
frost_target = lambda lg: torch.softmax(lg, 1)[:, 1]          # for binary fine-tuned models (class 1 = frost-damaged)

def run_cam(model, files, target, masks=None, bs=16):
    cams, probs = [], []
    for i in range(0, len(files), bs):
        ims = [prep(Image.open(f)) for f in files[i:i + bs]]
        x = torch.stack([norm(im) for im in ims]).to(DEVICE)
        c, p = gradcam(model, x, target); cams.append(c); probs.append(p)
    return np.concatenate(cams), np.concatenate(probs)
def attention_ratio(cam, mask):
    e = cam.sum(); return np.nan if e == 0 or mask.mean() == 0 else (cam[mask].sum() / e) / mask.mean()

In [ ]:
# 4. Load WEAPD models; Grad-CAM on EV3 (zero-shot) and on WEAPD test images of ice classes
MODELS = {}
for b in CFG["BACKBONES"]:
    m = timm.create_model(b, pretrained=False, num_classes=len(CLASSES)); sd = torch.load(CKPT[b], map_location="cpu")
    m.load_state_dict({k.replace("module.", ""): v for k, v in sd.items()}); MODELS[b] = m.to(DEVICE)
res = []; CAMS = {}
test = ev[ev.split == "test"].reset_index(drop=True)
for b, m in MODELS.items():
    cam, pr = run_cam(m, list(test.file), cold_target); CAMS[(b, "zero-shot")] = cam
    for i in range(len(test)):
        res.append(dict(backbone=b, stage="zero-shot", file=test.file[i], y=test.y[i], plant_frac=test.plant_frac[i],
                        cam_on_plant=float(cam[i][test["mask"][i]].sum() / (cam[i].sum() + 1e-8)), ratio=attention_ratio(cam[i], test["mask"][i]),
                        score=float(pr[i][COLD].sum())))
# WEAPD examples (frost / glaze / rime test images)
weapd_show = {}
if WEAPD_ROOT:
    t = wd[(wd.split == "test") & wd.label.isin(["frost", "glaze", "rime"])].groupby("label").head(2)
    real = {d.lower(): d for d in os.listdir(WEAPD_ROOT)}
    files = [os.path.join(WEAPD_ROOT, real.get(l, l), os.path.basename(str(s))) for l, s in zip(t.label, t.src)]
    files = [f for f in files if os.path.exists(f)]
    for b, m in MODELS.items():
        if files: weapd_show[b] = (files, run_cam(m, files, cold_target)[0])

In [ ]:
# 5. Fine-tune on EV3 pool (binary) from the WEAPD weights, then Grad-CAM of p(frost-damaged) on the same test images
tf_train = TV.Compose([TV.RandomCrop(S), TV.RandomHorizontalFlip(), TV.ToTensor(), TV.Normalize(MEAN, STD)])
pool = ev[ev.split == "pool"].reset_index(drop=True); cal = ev[ev.split == "cal"].reset_index(drop=True)
POOL_IMS = [Image.open(f).convert("RGB").resize((R, R), Image.BICUBIC) for f in pool.file]
from sklearn.metrics import roc_auc_score
for b in CFG["BACKBONES"]:
    torch.manual_seed(CFG["SEED"])
    m = timm.create_model(b, pretrained=False, num_classes=len(CLASSES)); m.load_state_dict(MODELS[b].state_dict()); m.reset_classifier(2); m = m.to(DEVICE)
    opt = torch.optim.AdamW(m.parameters(), lr=CFG["FT_LR"], weight_decay=0.05)
    idx = np.arange(len(pool)); steps = CFG["FT_EPOCHS"] * int(np.ceil(len(idx) / 32))
    sch = torch.optim.lr_scheduler.OneCycleLR(opt, CFG["FT_LR"], total_steps=steps)
    for ep in range(CFG["FT_EPOCHS"]):
        m.train(); np.random.shuffle(idx)
        for i in range(0, len(idx), 32):
            bi = idx[i:i + 32]; x = torch.stack([tf_train(POOL_IMS[k]) for k in bi]).to(DEVICE); y = torch.tensor(pool.y.values[bi]).to(DEVICE)
            opt.zero_grad(); F.cross_entropy(m(x), y).backward(); opt.step(); sch.step()
    cam, pr = run_cam(m, list(test.file), frost_target); CAMS[(b, "fine-tuned")] = cam
    print(b, "fine-tuned test AUROC", round(roc_auc_score(test.y, pr[:, 1]), 4))
    for i in range(len(test)):
        res.append(dict(backbone=b, stage="fine-tuned", file=test.file[i], y=test.y[i], plant_frac=test.plant_frac[i],
                        cam_on_plant=float(cam[i][test["mask"][i]].sum() / (cam[i].sum() + 1e-8)), ratio=attention_ratio(cam[i], test["mask"][i]),
                        score=float(pr[i][1])))
    del m; torch.cuda.empty_cache()

In [ ]:
# 6. Statistics: attention ratio zero-shot vs fine-tuned (paired Wilcoxon), by label
R_ = pd.DataFrame(res); R_.to_csv(os.path.join(CFG["OUT"], "gradcam_per_image.csv"), index=False)
summ = R_.groupby(["backbone", "stage", "y"]).agg(n=("ratio", "size"), plant_frac=("plant_frac", "mean"), cam_on_plant=("cam_on_plant", "mean"),
                                                  ratio_median=("ratio", "median"), ratio_mean=("ratio", "mean")).round(3).reset_index()
tests = []
for b in CFG["BACKBONES"]:
    a = R_[(R_.backbone == b) & (R_.stage == "zero-shot")].sort_values("file"); f = R_[(R_.backbone == b) & (R_.stage == "fine-tuned")].sort_values("file")
    ok = a.ratio.notna().values & f.ratio.notna().values
    tests.append(dict(backbone=b, n=int(ok.sum()), zero_shot_median=round(float(np.median(a.ratio.values[ok])), 3),
                      fine_tuned_median=round(float(np.median(f.ratio.values[ok])), 3), wilcoxon_p=wilcoxon(a.ratio.values[ok], f.ratio.values[ok]).pvalue,
                      pct_zero_shot_ratio_below_1=round(100 * float((a.ratio.values[ok] < 1).mean()), 1),
                      pct_fine_tuned_ratio_below_1=round(100 * float((f.ratio.values[ok] < 1).mean()), 1)))
summ.to_csv(os.path.join(CFG["OUT"], "gradcam_summary.csv"), index=False); pd.DataFrame(tests).to_csv(os.path.join(CFG["OUT"], "gradcam_tests.csv"), index=False)
print(summ.to_string()); print(pd.DataFrame(tests).to_string())

In [ ]:
# 7. Figures: (a) EV3 image, plant mask outline, zero-shot CAM, fine-tuned CAM; (b) WEAPD examples
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt, matplotlib.cm as cmx
def overlay(im, cam):
    h = cmx.jet(cam)[..., :3]; return (0.55 * np.asarray(im) / 255 + 0.45 * h).clip(0, 1)
b0 = CFG["BACKBONES"][0]
pick = list(test[test.y == 1].sample(CFG["N_SHOW"] // 2, random_state=1).index) + list(test[test.y == 0].sample(CFG["N_SHOW"] // 2, random_state=1).index)
fig, ax = plt.subplots(len(pick), 3, figsize=(6.3, 2.1 * len(pick)))
for r, i in enumerate(pick):
    im = prep(Image.open(test.file[i])); msk = test["mask"][i]
    ax[r, 0].imshow(im); ax[r, 0].contour(msk, levels=[0.5], colors="white", linewidths=0.8)
    ax[r, 1].imshow(overlay(im, CAMS[(b0, "zero-shot")][i])); ax[r, 2].imshow(overlay(im, CAMS[(b0, "fine-tuned")][i]))
    ax[r, 0].set_ylabel("frost-damaged" if test.y[i] else "healthy", fontsize=8)
    for c in range(3): ax[r, c].set_xticks([]); ax[r, c].set_yticks([])
for c, t in enumerate(["Image + plant mask", "Zero-shot (WEAPD)", "Fine-tuned on field"]): ax[0, c].set_title(t, fontsize=9)
plt.tight_layout(); plt.savefig(os.path.join(CFG["OUT"], "Figure_gradcam_ev3.png"), dpi=300); plt.close()
if b0 in weapd_show:
    files, cams = weapd_show[b0]
    fig, ax = plt.subplots(1, len(files), figsize=(2 * len(files), 2.2))
    for k, (f, c) in enumerate(zip(files, cams)):
        im = prep(Image.open(f)); ax[k].imshow(overlay(im, c)); ax[k].set_xticks([]); ax[k].set_yticks([])
        ax[k].set_title(os.path.basename(os.path.dirname(f)), fontsize=8)
    plt.tight_layout(); plt.savefig(os.path.join(CFG["OUT"], "Figure_gradcam_weapd.png"), dpi=300); plt.close()
# attention-ratio distribution
fig, ax = plt.subplots(figsize=(3.5, 2.6)); pos = 0; labels = []
for b in CFG["BACKBONES"]:
    for st in ["zero-shot", "fine-tuned"]:
        v = R_[(R_.backbone == b) & (R_.stage == st)].ratio.dropna().values
        q1, med, q3 = np.percentile(v, [25, 50, 75]); ax.plot([pos, pos], [q1, q3], color="k", lw=1); ax.plot([pos - .25, pos + .25], [med, med], color="k", lw=1.6)
        ax.scatter(pos + np.random.uniform(-.18, .18, len(v)), v, s=4, alpha=.35, color="#2a78d6" if st == "zero-shot" else "#eb6834", linewidths=0)
        labels.append(f"{b.split('_')[0]}\n{st}"); pos += 1
ax.axhline(1, color="#6b6b6b", lw=.8, ls=(0, (3, 3))); ax.set_xticks(range(pos), labels, fontsize=7); ax.set_ylabel("Attention ratio (plant)")
for s in ["top", "right"]: ax.spines[s].set_visible(False)
plt.tight_layout(); plt.savefig(os.path.join(CFG["OUT"], "Figure_gradcam_ratio.png"), dpi=300); plt.savefig(os.path.join(CFG["OUT"], "Figure_gradcam_ratio.pdf")); plt.close()

In [ ]:
# 8. Small zip
import zipfile
zp = "/kaggle/working/gradcam_results_small.zip"
with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
    for f in os.listdir(CFG["OUT"]): z.write(os.path.join(CFG["OUT"], f), f)
print(zp, "%.1f MB" % (os.path.getsize(zp) / 1e6), sorted(os.listdir(CFG["OUT"])))